# Saddle-Mediated Transport — Sigmoid Projection Along the Transit Axis

**Hypothesis.** Training dynamics pass through a saddle. The saddle axis is rotated from the principal components of the parameter trajectory. One end of the axis is the *wiggle-point* (the deflection right after first descent, which coincides with a peak in MLP dimensionality). The other end is the final checkpoint.

If that picture is right, projecting the full-parameter trajectory onto the unit vector `(terminal − wiggle)` should yield a sigmoid over epochs. Its derivative should peak at the saddle crossing.

**Reference variant.** `p109/s485/ds598` — reference healthy model with dense checkpointing (every 100 epochs through epoch 10000). `first_descent_end=1200`, `second_descent_onset=3584`, `grokking=5243`.

**Parameter space.** MLP only — `W_in` + `W_out` flattened and concatenated (the `COMPONENT_GROUPS['mlp']` bundle). Attention and embedding trajectories don't show the deflection, so scoping to MLP keeps the signal uncontaminated.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library.weights import COMPONENT_GROUPS
from miscope.visualization.renderers.dimensionality_dynamics import (
    _compute_rolling_trajectory_metrics,
)

FAMILY_NAME = 'modulo_addition_1layer'
family = load_family(FAMILY_NAME)

PRIME, SEED, DSEED = 109, 485, 598
variant = family.get_variant(prime=PRIME, seed=SEED, data_seed=DSEED)
label = f'p{PRIME}/s{SEED}/ds{DSEED}'

with open(variant.variant_dir / 'variant_summary.json') as f:
    summary = json.load(f)
TIMING = {
    'fd_end':  summary['first_descent_window']['end_epoch'],
    'sd_onset': summary.get('second_descent_onset_epoch'),
    'grok':     summary.get('test_loss_threshold_first_epoch'),
}
print(label, TIMING)

## 1. Load MLP parameter trajectory — full parameter space

`parameter_snapshot` stores all 9 weight matrices per epoch. For each checkpoint we flatten and concatenate `W_in` and `W_out` into a single vector, yielding `X` of shape `(n_epochs, d_mlp)`. No PCA reduction — the transit vector must live in the full space because the saddle axis is rotated from the principal directions.

In [ ]:
def load_mlp_param_trajectory(variant):
    """Load W_in ⊕ W_out flattened per epoch — full MLP parameter space."""
    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    epochs = sorted(loader.get_epochs('parameter_snapshot'))
    mlp_keys = COMPONENT_GROUPS['mlp']  # ['W_in', 'W_out']
    rows = []
    for e in epochs:
        snap = loader.load_epoch('parameter_snapshot', e)
        rows.append(np.concatenate([snap[k].flatten() for k in mlp_keys]))
    X = np.stack(rows).astype(np.float64)
    return np.array(epochs), X


epochs, X = load_mlp_param_trajectory(variant)
print(f'n_epochs = {len(epochs)}, d_mlp = {X.shape[1]}')
print(f'epoch range: {epochs[0]} → {epochs[-1]}')

## 2. Wiggle-point — peak of rolling PR₃ on the MLP trajectory

Apply `_compute_rolling_trajectory_metrics` (same function the dimensionality view uses) to the precomputed `mlp__projections`. Search for the PR₃ peak in the window `[fd_end, grok]`. Also compute the first PC3 extremum as a cross-check (your option 2).

In [ ]:
loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
pt = loader.load_cross_epoch('parameter_trajectory')
pt_epochs = pt['epochs']
mlp_proj = pt['mlp__projections']  # (n_epochs, 10)

pr3_mlp, ft_mlp = _compute_rolling_trajectory_metrics(mlp_proj, window=10)

# Peak in [fd_end, grok]
search_lo = TIMING['fd_end']
search_hi = TIMING['grok']
in_window = (pt_epochs >= search_lo) & (pt_epochs <= search_hi)
pr3_masked = np.where(in_window, pr3_mlp, -np.inf)
wiggle_idx_pr3 = int(np.argmax(pr3_masked))
wiggle_epoch_pr3 = int(pt_epochs[wiggle_idx_pr3])

# Cross-check: first PC3 extremum in same window (earliest |PC3| peak post-fd_end)
pc3 = mlp_proj[:, 2]
pc3_search = np.where(in_window, np.abs(pc3), -np.inf)
wiggle_idx_pc3 = int(np.argmax(pc3_search))
wiggle_epoch_pc3 = int(pt_epochs[wiggle_idx_pc3])

print(f'Rolling-PR₃ peak in [{search_lo}, {search_hi}] → epoch {wiggle_epoch_pr3}  (PR₃={pr3_mlp[wiggle_idx_pr3]:.3f})')
print(f'|PC3|    peak in [{search_lo}, {search_hi}] → epoch {wiggle_epoch_pc3}  (PC3={pc3[wiggle_idx_pc3]:+.3f})')

In [ ]:
# Visualize: rolling PR₃ and |PC3| with candidate wiggle-points
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=pt_epochs, y=pr3_mlp, mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pt_epochs, y=pc3, mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pr3, line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pc3, line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — wiggle-point candidates<br><sup>orange=fd_end · black=grok · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

## 3. Transit vector and scalar projection

For a chosen wiggle epoch `e_w` and terminal epoch `e_T`:

```
v = (X[e_T] − X[e_w]) / ‖X[e_T] − X[e_w]‖
s(t) = (X[t] − X[e_w]) · v     ≡     scalar distance along v from wiggle
```

Normalizing by `‖X[e_T] − X[e_w]‖` so `s=0` at wiggle and `s=1` at terminal. If the saddle picture is right, `s(t)` is sigmoidal with inflection at the crossing.

In [ ]:
def compute_transit_projection(X, epochs, wiggle_epoch, terminal_epoch):
    """Project full trajectory onto unit transit vector; normalize so wiggle→0, terminal→1."""
    i_w = int(np.where(epochs == wiggle_epoch)[0][0])
    i_T = int(np.where(epochs == terminal_epoch)[0][0])
    delta = X[i_T] - X[i_w]
    L = np.linalg.norm(delta)
    v_hat = delta / L
    # s normalized so s=0 at wiggle, s=1 at terminal
    s = (X - X[i_w]) @ v_hat / L
    return s, L, i_w, i_T


def compute_velocity(s, epochs):
    """ds/dt using central differences scaled by per-epoch gap."""
    s = np.asarray(s, dtype=np.float64)
    ep = np.asarray(epochs, dtype=np.float64)
    ds_dt = np.zeros_like(s)
    ds_dt[1:-1] = (s[2:] - s[:-2]) / (ep[2:] - ep[:-2])
    ds_dt[0]    = (s[1] - s[0]) / (ep[1] - ep[0])
    ds_dt[-1]   = (s[-1] - s[-2]) / (ep[-1] - ep[-2])
    return ds_dt


WIGGLE_EPOCH = wiggle_epoch_pr3           # primary candidate — rolling PR₃ peak
TERMINAL_EPOCH = int(epochs[-1])

s, L, i_w, i_T = compute_transit_projection(X, epochs, WIGGLE_EPOCH, TERMINAL_EPOCH)
v = compute_velocity(s, epochs)

saddle_idx = int(np.argmax(v))
saddle_epoch = int(epochs[saddle_idx])
print(f'wiggle epoch:     {WIGGLE_EPOCH}')
print(f'terminal epoch:   {TERMINAL_EPOCH}')
print(f'transit length:   {L:.3f}  (‖W_T − W_w‖ in MLP space)')
print(f'velocity-peak ep: {saddle_epoch}  (ds/dt max = {v[saddle_idx]:.2e})')
print(f's at wiggle = {s[i_w]:.3f}, s at terminal = {s[i_T]:.3f}  (should be 0.0 and 1.0)')

In [ ]:
# Sigmoid + velocity plot
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  scalar projection along transit axis  (0=wiggle, 1=terminal)',
                                    'ds/dt  —  projection velocity'])

fig.add_trace(go.Scatter(x=epochs, y=s, mode='lines+markers', name='s(t)',
                         line=dict(color='#1f77b4', width=2),
                         marker=dict(size=4)), row=1, col=1)
fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs, y=v, mode='lines+markers', name='ds/dt',
                         line=dict(color='#d62728', width=2),
                         marker=dict(size=4)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'],    line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['sd_onset'],  line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],      line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=WIGGLE_EPOCH,        line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=saddle_epoch,        line=dict(color='green',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{label} — transit projection  (wiggle ep={WIGGLE_EPOCH}, terminal ep={TERMINAL_EPOCH})<br>'
           '<sup>orange=fd_end · teal=sd_onset · black=grok · red=wiggle · green=ds/dt peak</sup>'),
    template='plotly_white', height=700, width=1000, showlegend=False,
)
fig.show()

## 4. Wiggle-point sensitivity — PR₃ peak vs |PC3| peak

Overlay `s(t)` for both wiggle-point choices against the same terminal. If both give the same sigmoid shape (up to an affine rescaling of the axis), the two operationalizations agree on the saddle picture.

In [ ]:
WIGGLE_CANDIDATES = [
    ('PR₃ peak',   wiggle_epoch_pr3, '#1f77b4'),
    ('|PC3| peak', wiggle_epoch_pc3, '#9467bd'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per wiggle-point choice',
                                    'ds/dt per wiggle-point choice'])

for name, we, color in WIGGLE_CANDIDATES:
    s_c, L_c, _, _ = compute_transit_projection(X, epochs, we, TERMINAL_EPOCH)
    v_c = compute_velocity(s_c, epochs)
    fig.add_trace(go.Scatter(x=epochs, y=s_c, mode='lines', name=f'{name} (ep {we})',
                             line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=v_c, mode='lines', name=f'{name} (ep {we})',
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)

fig.update_yaxes(title_text='s', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — wiggle-point sensitivity  (terminal ep={TERMINAL_EPOCH})',
    template='plotly_white', height=700, width=1000,
)
fig.show()

## 5. Terminal sensitivity — does the sigmoid shift under truncation?

If MLP dimensionality is cyclic (your hypothesis), the choice of terminal may distort the transit axis. Overlay `s(t)` for terminal choices at 10k, 15k, and final (25k) — using the PR₃-peak wiggle. A stable sigmoid under all three says the saddle axis is robust; drift says we need to re-examine the terminal.

In [ ]:
def closest_epoch(epochs_arr, target):
    return int(epochs_arr[int(np.argmin(np.abs(epochs_arr - target)))])

TERMINAL_CANDIDATES = [
    ('terminal 10k',  closest_epoch(epochs, 10000)),
    ('terminal 15k',  closest_epoch(epochs, 15000)),
    ('terminal final', int(epochs[-1])),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) per terminal choice  (wiggle = PR₃ peak)',
                                    'ds/dt per terminal choice'])
colors = ['#2ca02c', '#ff7f0e', '#d62728']

for (name, te), color in zip(TERMINAL_CANDIDATES, colors):
    s_c, L_c, _, _ = compute_transit_projection(X, epochs, wiggle_epoch_pr3, te)
    v_c = compute_velocity(s_c, epochs)
    fig.add_trace(go.Scatter(x=epochs, y=s_c, mode='lines',
                             name=f'{name} (ep {te}, L={L_c:.2f})',
                             line=dict(color=color, width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=v_c, mode='lines', name=name,
                             line=dict(color=color, width=2), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=wiggle_epoch_pr3, line=dict(color='red',    dash='solid', width=1), row=row, col=1)

fig.update_yaxes(title_text='s', row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — terminal sensitivity  (wiggle ep={wiggle_epoch_pr3})',
    template='plotly_white', height=700, width=1000,
)
fig.show()

## 6. Checkpointing resilience — subsample test

Take p109's dense 100-epoch trajectory and subsample it to mimic what sparse-checkpointed variants look like. Two scenarios tested:

**(a) Uniformly sparse** — every-500 spacing throughout. Matches the typical 'plateau-sparse' pattern on existing variants.

**(b) Hybrid** — dense (every 100) through a cutoff, sparse (every 500) after. Matches what's realistic for new training runs: dense checkpointing through the grokking region, then sparse post-grok.

For each pattern we refit PCA on the subsampled snapshots (simulating what the `parameter_trajectory` analyzer would have stored) and re-run the full pipeline. Metrics tracked vs dense baseline: wiggle epoch shift, saddle epoch shift, transit length Δ.

In [ ]:
from sklearn.decomposition import PCA


def identify_wiggle(X, epochs_arr, fd_end, grok):
    """Refit PCA, compute rolling PR₃, find peak in [fd_end, grok]."""
    pca = PCA(n_components=min(10, len(X) - 1)).fit(X)
    proj = pca.transform(X)
    pr3, _ = _compute_rolling_trajectory_metrics(proj, window=min(10, len(X)))
    in_win = (epochs_arr >= fd_end) & (epochs_arr <= grok)
    if in_win.sum() == 0:
        return None
    return int(epochs_arr[int(np.argmax(np.where(in_win, pr3, -np.inf)))])


def full_pipeline(X, eps, fd_end, grok):
    we = identify_wiggle(X, eps, fd_end, grok)
    if we is None:
        return None
    s, L, _, _ = compute_transit_projection(X, eps, we, int(eps[-1]))
    v = compute_velocity(s, eps)
    return dict(
        wiggle=we, L=L,
        saddle=int(eps[int(np.argmax(v))]),
        max_dsdt=float(v.max()),
        s=s, v=v, eps=eps,
    )


DENSE = full_pipeline(X, epochs, TIMING['fd_end'], TIMING['grok'])
patterns = {'dense (every 100)': DENSE}

for stride in [500, 1000]:
    mask = (epochs % stride == 0)
    mask[0] = True; mask[-1] = True
    patterns[f'uniform sparse ({stride})'] = full_pipeline(
        X[mask], epochs[mask], TIMING['fd_end'], TIMING['grok']
    )

for cutoff in [3000, 4000, 6000]:
    mask = ((epochs <= cutoff) & (epochs % 100 == 0)) | ((epochs > cutoff) & (epochs % 500 == 0))
    mask[0] = True; mask[-1] = True
    patterns[f'hybrid dense<={cutoff}, sparse 500 after'] = full_pipeline(
        X[mask], epochs[mask], TIMING['fd_end'], TIMING['grok']
    )

print(f'{"pattern":40s}  {"n":>4s}  {"wiggle":>7s}  {"Δwig":>6s}  {"saddle":>7s}  {"Δsad":>6s}  {"L":>6s}  {"ΔL":>6s}')
print('-' * 90)
for name, r in patterns.items():
    dw = r['wiggle'] - DENSE['wiggle']
    ds = r['saddle'] - DENSE['saddle']
    dL = r['L'] - DENSE['L']
    print(f'{name:40s}  {len(r["eps"]):4d}  {r["wiggle"]:7d}  {dw:+6d}  {r["saddle"]:7d}  {ds:+6d}  {r["L"]:6.2f}  {dL:+6.2f}')


In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) — dense vs sparse patterns',
                                    'ds/dt — dense vs sparse patterns'])
palette = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c', '#9467bd', '#17becf']

for (name, r), color in zip(patterns.items(), palette):
    fig.add_trace(go.Scatter(x=r['eps'], y=r['s'], mode='lines+markers',
                             name=name, line=dict(color=color, width=2),
                             marker=dict(size=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=r['eps'], y=r['v'], mode='lines+markers',
                             name=name, line=dict(color=color, width=2),
                             marker=dict(size=4), showlegend=False), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=TIMING['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=TIMING['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)

fig.update_yaxes(title_text='s',     row=1, col=1)
fig.update_yaxes(title_text='ds/dt', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{label} — checkpointing sensitivity  (baseline = dense every 100 epochs)',
    template='plotly_white', height=750, width=1100,
)
fig.show()

---

## 7. Broadening — p113/s999/ds598 (canon)

User extended dense (every-100) checkpointing through epoch **15000** on p113/s999/ds598. Reference timing for p113: `fd_end=1200`, `sd_onset=9503`, `grok=12448`, `sd_end=12400`. That's grok + ~21% margin and ~+1k past second-descent end — comfortably above the empirical rule from §6.

Predictions if the saddle-transport picture generalizes:
1. Clean sigmoid `s(t)`.
2. `ds/dt` peak in late plateau / early second descent (i.e., **before** grok), echoing p109's ~950-epoch pre-grok crossing.
3. Terminal-robust under truncation.

We refactor the pipeline into `run_saddle_pipeline(variant)` so this and future variants reuse one function.

In [ ]:
def run_saddle_pipeline(variant, terminal=None):
    """Full saddle-transport pipeline for a variant.

    For non-grokkers (`test_loss_threshold_first_epoch` ∈ {-1, None}), the wiggle
    search window falls back to [fd_end, sd_onset] — the structural analog of
    [fd_end, grok], since 'commitment' for these models lives inside / after
    second descent rather than at a grok point.

    `terminal` defaults to the final checkpoint; pass an explicit epoch to study
    alternative landing points (e.g. 'near-landing' for rebounders).
    """
    p = variant.params
    label = f'p{p["prime"]}/s{p["seed"]}/ds{p["data_seed"]}'
    with open(variant.variant_dir / 'variant_summary.json') as f:
        summary = json.load(f)
    grok_raw = summary.get('test_loss_threshold_first_epoch')
    grok = grok_raw if (grok_raw not in (None, -1)) else None
    timing = {
        'fd_end':   summary['first_descent_window']['end_epoch'],
        'sd_onset': summary.get('second_descent_onset_epoch'),
        'sd_end':   summary.get('second_descent_window', {}).get('end_epoch'),
        'grok':     grok,
    }

    epochs, X = load_mlp_param_trajectory(variant)

    loader = ArtifactLoader(str(variant.variant_dir / 'artifacts'))
    pt = loader.load_cross_epoch('parameter_trajectory')
    pt_eps = pt['epochs']
    mlp_proj = pt['mlp__projections']
    pr3, _ = _compute_rolling_trajectory_metrics(mlp_proj, window=10)
    pc3 = mlp_proj[:, 2]

    # Wiggle search window: [fd_end, grok] if grok exists, else [fd_end, sd_onset]
    search_hi = grok if grok is not None else timing['sd_onset']
    in_win = (pt_eps >= timing['fd_end']) & (pt_eps <= search_hi)
    we_pr3 = int(pt_eps[int(np.argmax(np.where(in_win, pr3, -np.inf)))])
    we_pc3 = int(pt_eps[int(np.argmax(np.where(in_win, np.abs(pc3), -np.inf)))])

    if terminal is None:
        terminal = int(epochs[-1])
    s, L, i_w, i_T = compute_transit_projection(X, epochs, we_pr3, terminal)
    v = compute_velocity(s, epochs)
    saddle = int(epochs[int(np.argmax(v))])

    return dict(
        label=label, variant=variant, summary=summary, timing=timing,
        epochs=epochs, X=X,
        pt_eps=pt_eps, mlp_proj=mlp_proj, pr3=pr3, pc3=pc3,
        wiggle_pr3=we_pr3, wiggle_pc3=we_pc3,
        terminal=terminal, s=s, v=v, L=L, saddle=saddle,
        max_dsdt=float(v.max()),
        min_dsdt=float(v.min()),
        s_min=float(s.min()), s_min_ep=int(epochs[int(np.argmin(s))]),
        s_max=float(s.max()), s_max_ep=int(epochs[int(np.argmax(s))]),
    )


variant_113 = family.get_variant(prime=113, seed=999, data_seed=598)
R113 = run_saddle_pipeline(variant_113)

print(f'{R113["label"]}  ({R113["epochs"][0]} → {R113["epochs"][-1]}, n={len(R113["epochs"])})')
print(f'  timing       : {R113["timing"]}')
print(f'  wiggle (PR₃) : ep {R113["wiggle_pr3"]}  (PR₃={R113["pr3"][np.argmax(R113["pr3"])]:.3f} max overall)')
print(f'  wiggle (|PC3|): ep {R113["wiggle_pc3"]}')
print(f'  transit L    : {R113["L"]:.3f}')
print(f'  saddle ep    : {R113["saddle"]}  (max ds/dt = {R113["max_dsdt"]:.2e})')
print(f'  saddle − grok: {R113["saddle"] - R113["timing"]["grok"]:+d} epochs')

In [ ]:
# Wiggle candidates for p113 — same layout as §2 plot for direct comparison
R = R113
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pr3'], mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pc3'], mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'], line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['grok'],   line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'], line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['wiggle_pc3'], line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R["label"]} — wiggle-point candidates<br><sup>orange=fd_end · teal=sd_onset · black=grok · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

In [ ]:
# Sigmoid + velocity for p113 — same layout as §3
R = R113
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  scalar projection along transit axis  (0=wiggle, 1=terminal)',
                                    'ds/dt  —  projection velocity'])

fig.add_trace(go.Scatter(x=R['epochs'], y=R['s'], mode='lines+markers', name='s(t)',
                         line=dict(color='#1f77b4', width=2),
                         marker=dict(size=4)), row=1, col=1)
fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.add_trace(go.Scatter(x=R['epochs'], y=R['v'], mode='lines+markers', name='ds/dt',
                         line=dict(color='#d62728', width=2),
                         marker=dict(size=4)), row=2, col=1)

for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'],   line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['grok'],     line=dict(color='black',  dash='dash', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'],         line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['saddle'],             line=dict(color='green',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{R["label"]} — transit projection  (wiggle ep={R["wiggle_pr3"]}, terminal ep={R["terminal"]})<br>'
           '<sup>orange=fd_end · teal=sd_onset · black=grok · red=wiggle · green=ds/dt peak</sup>'),
    template='plotly_white', height=700, width=1000, showlegend=False,
)
fig.show()

---

## 8. Falsifier — p101/s485/ds999 (rebounder)

p101/s485/ds999 enters second descent (sd_onset=6463), reaches near-zero test loss at sd_end=9300, then **rebounds out** — final test loss = 0.049. No grok.

Two terminal probes — designed to ask different questions:

- **Terminal = final (24999).** Healthy-pipeline analog. The transit axis points to the rebounded final state. Question: does the trajectory look monotonically sigmoidal toward this terminal, or does it reveal non-monotonic structure?

- **Terminal = near-landing (sd_end=9300).** "Where the model nearly landed." The transit axis points to the bottom of second descent (the model's transient minimum). Question: does the trajectory overshoot 1.0 (= excursion past the near-landing point in MLP-parameter space), and if so, does it return?

The pipeline now falls back to `[fd_end, sd_onset]` for the wiggle search window when no grok exists, so this works without ad-hoc edits.

In [ ]:
variant_101r = family.get_variant(prime=101, seed=485, data_seed=999)

# Pick near-landing epoch from the test-loss minimum inside the second-descent window
md_101r = variant_101r.metadata
test_losses_101r = np.array(md_101r['test_losses'])
with open(variant_101r.variant_dir / 'variant_summary.json') as f:
    summary_101r = json.load(f)
sd_lo = summary_101r['second_descent_window']['start_epoch']
sd_hi = summary_101r['second_descent_window']['end_epoch']
near_land_raw = int(sd_lo + np.argmin(test_losses_101r[sd_lo:sd_hi + 1]))
near_land_loss = float(test_losses_101r[near_land_raw])
final_loss_101r = float(test_losses_101r[-1])

# Snap to nearest checkpoint (dense-100 covers this region)
loader_101r = ArtifactLoader(str(variant_101r.variant_dir / 'artifacts'))
checkpoint_eps = np.array(sorted(loader_101r.get_epochs('parameter_snapshot')))
near_land_ep = int(checkpoint_eps[np.argmin(np.abs(checkpoint_eps - near_land_raw))])

R101_final = run_saddle_pipeline(variant_101r)
R101_land  = run_saddle_pipeline(variant_101r, terminal=near_land_ep)

print(f'p101/s485/ds999 rebounder')
print(f'  test loss: min={near_land_loss:.5f} at ep {near_land_raw} (snapped {near_land_ep})  ·  final={final_loss_101r:.5f}')
print(f'  timing: {R101_final["timing"]}  (grok=None ⇒ wiggle window = [fd_end, sd_onset])')
print(f'  wiggle (PR₃) : ep {R101_final["wiggle_pr3"]}  (PR₃={R101_final["pr3"][np.argmax(R101_final["pr3"])]:.3f} max overall)')
print(f'  wiggle (|PC3|): ep {R101_final["wiggle_pc3"]}')
print()
for tag, R in [('terminal=final', R101_final), (f'terminal=near-land({near_land_ep})', R101_land)]:
    print(f'  --- {tag} ---')
    print(f'    L = {R["L"]:.3f}')
    print(f'    saddle (max ds/dt) = ep {R["saddle"]}  (max ds/dt = {R["max_dsdt"]:.2e}, min ds/dt = {R["min_dsdt"]:.2e})')
    print(f'    s overshoot: max s = {R["s_max"]:+.4f} at ep {R["s_max_ep"]}  (>1.0 ⇒ trajectory passes the terminal)')
    print(f'    s_min pre-wiggle = {R["s_min"]:+.4f} at ep {R["s_min_ep"]}')


In [ ]:
# Wiggle candidates — p101 rebounder
R = R101_final
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['MLP trajectory — rolling PR₃',
                                    'MLP trajectory — PC3 coordinate'])
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pr3'], mode='lines', name='PR₃',
                         line=dict(color='#d62728', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=R['pt_eps'], y=R['pc3'], mode='lines', name='PC3',
                         line=dict(color='#1f77b4', width=2)), row=2, col=1)

# No grok line — but mark sd_end (the rebound 'reset' point) in addition to sd_onset
for row in (1, 2):
    fig.add_vline(x=R['timing']['fd_end'],   line=dict(color='orange', dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_onset'], line=dict(color='teal',   dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['timing']['sd_end'],   line=dict(color='magenta',dash='dot', width=1), row=row, col=1)
    fig.add_vline(x=R['wiggle_pr3'], line=dict(color='red',    dash='solid', width=1.5), row=row, col=1)
    fig.add_vline(x=R['wiggle_pc3'], line=dict(color='purple', dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PC3', row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=f'{R["label"]} — wiggle-point candidates (rebounder)<br><sup>orange=fd_end · teal=sd_onset · magenta=sd_end (near-landing) · red=PR₃ peak · purple=|PC3| peak</sup>',
    template='plotly_white', height=600, width=1000,
)
fig.show()

In [ ]:
# Side-by-side: terminal=final vs terminal=near-land. The interesting comparison —
# whether rebound shows as overshoot in the near-land projection.
PROBES = [('terminal=final', R101_final, '#1f77b4'),
          (f'terminal=near-land(ep {near_land_ep})', R101_land, '#d62728')]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t) — both terminal probes (s=1.0 line is the near-landing terminal)',
                                    'ds/dt — both terminal probes'])

for tag, R, color in PROBES:
    fig.add_trace(go.Scatter(x=R['epochs'], y=R['s'], mode='lines+markers',
                             name=f'{tag}  (L={R["L"]:.2f})',
                             line=dict(color=color, width=2),
                             marker=dict(size=4)), row=1, col=1)
    fig.add_trace(go.Scatter(x=R['epochs'], y=R['v'], mode='lines+markers',
                             name=tag, line=dict(color=color, width=2),
                             marker=dict(size=4), showlegend=False), row=2, col=1)

# Annotate the s_max overshoot for the near-land probe
R = R101_land
fig.add_annotation(x=R['s_max_ep'], y=R['s_max'], xref='x1', yref='y1',
                   text=f'overshoot: s={R["s_max"]:.3f} at ep {R["s_max_ep"]}',
                   showarrow=True, arrowhead=2, ax=40, ay=-30,
                   bgcolor='rgba(255,255,255,0.85)', bordercolor='#d62728',
                   row=1, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)

T = R101_final['timing']
for row in (1, 2):
    fig.add_vline(x=T['fd_end'],   line=dict(color='orange',  dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=T['sd_onset'], line=dict(color='teal',    dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=T['sd_end'],   line=dict(color='magenta', dash='dot',   width=1), row=row, col=1)
    fig.add_vline(x=R101_final['wiggle_pr3'], line=dict(color='red',  dash='solid', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='s (normalized)', row=1, col=1)
fig.update_yaxes(title_text='ds/dt',         row=2, col=1)
fig.update_xaxes(title_text='epoch', row=2, col=1)
fig.update_layout(
    title=(f'{R101_final["label"]} — rebounder transit projection (two terminals)<br>'
           '<sup>orange=fd_end · teal=sd_onset · magenta=sd_end (near-landing) · red=wiggle</sup>'),
    template='plotly_white', height=750, width=1100,
)
fig.show()

---

## 9. Cross-variant comparison — p109 / p113 / p101-rebounder

Refined dimensionless health-signature candidates (after observing two healthy variants):

- `(saddle − commitment) / commitment` — fractional offset, comparable across timescales
- `|s_min|` — retreat depth pre-wiggle
- `max(ds/dt) × commitment` — grok/commitment-rescaled crossing sharpness

For grokkers, `commitment = grok`. For non-grokkers (rebounders/diverged), `commitment = sd_end` (the near-landing or "where it nearly committed" epoch). This keeps the fractional offset interpretable across variant classes.

Below: full comparison including p101 rebounder with `terminal=final` (apples-to-apples). The rebounder-specific signature — overshoot under `terminal=near-landing` — is in §8, not in this table.

In [ ]:
# Re-bundle p109 results via the same pipeline function so comparison is apples-to-apples
R109 = run_saddle_pipeline(variant)  # variant is p109 from §1
RUNS = [R109, R113, R101_final]      # p101 with terminal=final for parity with healthy variants

# For non-grokkers (p101), use sd_end (near-landing epoch) as the 'commitment' anchor,
# so we can report (saddle - commitment)/commitment as a unified ratio.
def commitment_epoch(R):
    return R['timing']['grok'] if R['timing']['grok'] is not None else R['timing']['sd_end']

def commitment_label(R):
    return 'grok' if R['timing']['grok'] is not None else 'sd_end'

print(f'{"variant":20s}  {"d_mlp":>6s}  {"commit":>10s}  {"wiggle":>7s}  {"saddle":>7s}  '
      f'{"sad-com":>8s}  {"frac":>6s}  {"L":>7s}  {"|s_min|":>8s}  {"max·com":>8s}')
print('-' * 105)
for R in RUNS:
    c = commitment_epoch(R); cl = commitment_label(R)
    sg = R['saddle'] - c
    frac = sg / c
    smin = abs(R['s_min'])
    print(f'{R["label"]:20s}  {R["X"].shape[1]:6d}  {f"{c} ({cl})":>10s}  '
          f'{R["wiggle_pr3"]:7d}  {R["saddle"]:7d}  {sg:+8d}  {frac:+6.2f}  '
          f'{R["L"]:7.2f}  {smin:8.3f}  {R["max_dsdt"]*c:8.2f}')

In [ ]:
# Overlay s(t) and ds/dt on commitment-normalized x-axis (epoch / commitment_epoch)
# - commitment = grok for healthy variants, sd_end for rebounders/non-grokkers
# - lets us compare saddle structure across variants with very different timescales
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['s(t)  —  normalized by commitment epoch',
                                    'ds/dt × commitment  —  velocity rescaled to commitment-time'])
colors = {'p109/s485/ds598': '#1f77b4',
          'p113/s999/ds598': '#d62728',
          'p101/s485/ds999': '#2ca02c'}

for R in RUNS:
    c = commitment_epoch(R)
    x_norm = R['epochs'] / c
    color = colors[R['label']]
    legend_name = f'{R["label"]} (commit={commitment_label(R)}={c})'
    fig.add_trace(go.Scatter(x=x_norm, y=R['s'], mode='lines+markers',
                             name=legend_name, line=dict(color=color, width=2),
                             marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_norm, y=R['v'] * c, mode='lines+markers',
                             name=R['label'], line=dict(color=color, width=2),
                             marker=dict(size=3), showlegend=False), row=2, col=1)
    fig.add_vline(x=R['wiggle_pr3'] / c,
                  line=dict(color=color, dash='dot', width=1), row=1, col=1, opacity=0.4)
    fig.add_vline(x=R['saddle'] / c,
                  line=dict(color=color, dash='solid', width=1), row=2, col=1, opacity=0.4)

for row in (1, 2):
    fig.add_vline(x=1.0, line=dict(color='black', dash='dash', width=1), row=row, col=1)

fig.add_hline(y=0.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot', width=1), row=1, col=1)
fig.add_hline(y=0.5, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1), row=1, col=1)

fig.update_yaxes(title_text='s (normalized)',  row=1, col=1)
fig.update_yaxes(title_text='ds/dt × commit',  row=2, col=1)
fig.update_xaxes(title_text='epoch / commitment_epoch', row=2, col=1)
fig.update_layout(
    title='p109 / p113 / p101-rebounder — transit projection on commitment-normalized time<br>'
          '<sup>black dash = commitment (x=1) · colored dots = each run\'s wiggle (top) and saddle (bottom)</sup>',
    template='plotly_white', height=750, width=1100,
)
fig.show()